# Intraday Curve Building & Swap Pricing

## Real-time SOFR Curve Construction and Trade Mark-to-Market

This notebook provides tools for:

1. **Intraday SOFR Curve Building** - Build curves from live market data
2. **SDR Trade Pricing** - Mark SDR trades to the curve
3. **Rich/Cheap Analysis** - Compare traded rates vs fair value
4. **Curve Timeseries** - Track curve evolution intraday
5. **Risk Analytics** - Calculate PV01, DV01 for SDR trades

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.rl_usd_sofr_mt_builder import rl_usd_sofr_mt_builder
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.stir_curve_building_utils import get_fixings

# Initialize SDR Data Builder
cache_path = r"/tmp/sdr_cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

## 1. Build SOFR Curve from Market Data

In [ ]:
# Fetch SOFR fixings
sofr_fixings = get_fixings(rate="sofr", n=90)
print(f"SOFR Fixings loaded: {len(sofr_fixings)} days")
print(f"Latest SOFR fixing: {sofr_fixings.index[-1]} = {sofr_fixings.iloc[-1]:.4%}")
sofr_fixings.tail()

In [ ]:
# Build the SOFR curve
# Options for 'snap': datetime for historical, 'live' for real-time quotes
curve_timestamp = NY_tz.localize(datetime.datetime(2025, 12, 19, 15, 0))

print(f"Building curve as of: {curve_timestamp}")

curve = rl_usd_sofr_mt_builder(
    curve_id="usd_sofr_mt",
    snap=curve_timestamp,  # Use 'live' for real-time
    sofr_fixings=sofr_fixings,
    n_ser_contracts=12,      # Number of 1M SOFR contracts
    n_sfr_contracts=12,      # Number of 3M SOFR contracts
    n_plus_fomc_years=2,     # Years of FOMC meetings to include
    live_side="mid",
    medium_term_tenors=["5Y", "7Y", "10Y", "20Y", "30Y"],
    max_tenor="30Y",
    extrapolation_yrs=30
)

In [ ]:
# Visualize the curve
curve.rl_pricing_curve.plot("1d")

In [ ]:
# Display curve instruments and their rates
print("Curve Instruments:")
print("="*60)
for name, inst in curve.rl_pricing_curve_instruments.items():
    if hasattr(inst, 'rate'):
        rate = inst.rate(curves=curve.rl_pricing_curve)
        print(f"{name:20s}: {rate*100:.4f}%")

## 2. Fetch SDR Trades and Calculate Fair Values

In [ ]:
# Fetch SDR trades for the analysis period
start = NY_tz.localize(datetime.datetime(2025, 12, 19, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 19, 17, 0))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start,
    end_timestamp=end,
    agency="CFTC",
    asset_class="RATES"
)

print(f"Total trades fetched: {len(raw_df):,}")

In [ ]:
def filter_vanilla_sofr_swaps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter to vanilla USD SOFR OIS swaps suitable for pricing.
    """
    df = df.copy()
    
    # New trades only
    df = df[df['Action type'] == 'NEWT']
    
    # USD currency
    df = df[df['Notional currency-Leg 1'] == 'USD']
    
    # SOFR-based swaps
    sofr_mask = df['UPI Underlier Name'].str.contains('SOFR', case=False, na=False)
    df = df[sofr_mask]
    
    # OIS swaps (exclude swaptions, caps/floors)
    ois_mask = df['UPI FISN'].str.contains('OIS|Swap', case=False, na=False)
    option_mask = df['UPI FISN'].str.contains('Call|Put|Cap|Floor|Swaption', case=False, na=False)
    df = df[ois_mask & ~option_mask]
    
    # Must have a valid fixed rate
    df['Fixed rate-Leg 1'] = pd.to_numeric(df['Fixed rate-Leg 1'], errors='coerce')
    df = df[df['Fixed rate-Leg 1'].notna() & (df['Fixed rate-Leg 1'] > 0)]
    
    # Parse dates and notionals
    df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')
    df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], errors='coerce')
    df = df[df['Effective Date'].notna() & df['Expiration Date'].notna()]
    
    # Calculate tenor
    df['Tenor_Days'] = (df['Expiration Date'] - df['Effective Date']).dt.days
    df['Tenor_Years'] = df['Tenor_Days'] / 365.25
    
    # Filter reasonable tenors (3 months to 50 years)
    df = df[(df['Tenor_Years'] >= 0.25) & (df['Tenor_Years'] <= 50)]
    
    # Parse notional
    df['Notional amount-Leg 1'] = df['Notional amount-Leg 1'].astype(str).str.replace(',', '')
    df['Notional amount-Leg 1'] = pd.to_numeric(df['Notional amount-Leg 1'], errors='coerce')
    
    # Standard IMM or spot start trades
    # For simplicity, we'll use trades with spot start (T+2)
    
    return df.reset_index(drop=True)

vanilla_df = filter_vanilla_sofr_swaps(raw_df)
print(f"Vanilla SOFR swaps: {len(vanilla_df):,}")

In [ ]:
def price_sdr_swap(row, pricing_curve: rl.Curve, curve_id: str = "usd_sofr_mt") -> dict:
    """
    Price an SDR swap trade against the curve.
    Returns fair rate, NPV, PV01, and rich/cheap analysis.
    """
    try:
        effective = row['Effective Date']
        expiration = row['Expiration Date']
        traded_rate = row['Fixed rate-Leg 1']
        notional = row['Notional amount-Leg 1'] if pd.notna(row['Notional amount-Leg 1']) else 10_000_000
        
        # Create the swap using rateslib
        swap = rl.IRS(
            effective=effective,
            termination=expiration,
            notional=notional,
            fixed_rate=traded_rate,
            spec="usd_irs",
            curves=curve_id
        )
        
        # Calculate metrics
        fair_rate = swap.rate(curves=pricing_curve)
        npv = swap.npv(curves=pricing_curve)
        pv01 = swap.analytic_delta(curve=pricing_curve)
        
        # Rich/cheap in bps
        rich_cheap_bps = (traded_rate - fair_rate) * 10000
        
        return {
            'fair_rate': fair_rate,
            'npv': npv,
            'pv01': pv01,
            'rich_cheap_bps': rich_cheap_bps,
            'error': None
        }
    except Exception as e:
        return {
            'fair_rate': np.nan,
            'npv': np.nan,
            'pv01': np.nan,
            'rich_cheap_bps': np.nan,
            'error': str(e)
        }

# Price all vanilla swaps
pricing_results = []
for idx, row in vanilla_df.iterrows():
    result = price_sdr_swap(row, curve.rl_pricing_curve)
    result['Dissemination Identifier'] = row['Dissemination Identifier']
    pricing_results.append(result)

pricing_df = pd.DataFrame(pricing_results)
pricing_df = pricing_df.merge(vanilla_df[['Dissemination Identifier', 'Effective Date', 'Expiration Date', 
                                           'Tenor_Years', 'Fixed rate-Leg 1', 'Notional amount-Leg 1',
                                           'Event timestamp']], 
                               on='Dissemination Identifier')

# Filter successful pricings
priced_df = pricing_df[pricing_df['error'].isna()].copy()
print(f"Successfully priced: {len(priced_df):,} of {len(vanilla_df):,} trades")

## 3. Rich/Cheap Analysis

In [ ]:
if len(priced_df) > 0:
    # Scatter plot of rich/cheap by tenor
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=priced_df['Tenor_Years'],
        y=priced_df['rich_cheap_bps'],
        mode='markers',
        marker=dict(
            size=8,
            color=priced_df['rich_cheap_bps'],
            colorscale='RdYlGn_r',
            colorbar=dict(title='Rich/Cheap (bps)'),
            line=dict(width=1, color='black')
        ),
        text=[f"Rate: {r*100:.3f}%<br>Fair: {f*100:.3f}%<br>Notional: ${n:,.0f}" 
              for r, f, n in zip(priced_df['Fixed rate-Leg 1'], priced_df['fair_rate'], priced_df['Notional amount-Leg 1'])],
        hoverinfo='text+x+y'
    ))
    
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    
    fig.update_layout(
        title='Rich/Cheap Analysis: Traded Rate vs Fair Value',
        xaxis_title='Tenor (Years)',
        yaxis_title='Rich/Cheap (bps)',
        height=600
    )
    fig.show()
else:
    print("No trades successfully priced.")

In [ ]:
if len(priced_df) > 0:
    # Histogram of rich/cheap distribution
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=priced_df['rich_cheap_bps'],
        nbinsx=50,
        marker_color='steelblue',
        opacity=0.75
    ))
    
    fig.add_vline(x=0, line_dash='dash', line_color='red', line_width=2)
    fig.add_vline(x=priced_df['rich_cheap_bps'].mean(), line_dash='dot', line_color='green', line_width=2)
    
    fig.update_layout(
        title=f"Rich/Cheap Distribution (Mean: {priced_df['rich_cheap_bps'].mean():.2f} bps)",
        xaxis_title='Rich/Cheap (bps)',
        yaxis_title='Count',
        height=400
    )
    fig.show()

In [ ]:
if len(priced_df) > 0:
    # Rich/cheap by tenor bucket
    def assign_tenor_bucket(years):
        if years <= 1:
            return 'Front End (<1Y)'
        elif years <= 2:
            return '1-2Y'
        elif years <= 5:
            return '2-5Y'
        elif years <= 10:
            return '5-10Y'
        elif years <= 20:
            return '10-20Y'
        else:
            return '20Y+'
    
    priced_df['Tenor_Bucket'] = priced_df['Tenor_Years'].apply(assign_tenor_bucket)
    
    tenor_order = ['Front End (<1Y)', '1-2Y', '2-5Y', '5-10Y', '10-20Y', '20Y+']
    
    fig = go.Figure()
    
    for bucket in tenor_order:
        bucket_data = priced_df[priced_df['Tenor_Bucket'] == bucket]['rich_cheap_bps']
        if len(bucket_data) > 0:
            fig.add_trace(go.Box(
                y=bucket_data,
                name=bucket,
                boxpoints='all',
                jitter=0.3,
                pointpos=-1.8
            ))
    
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    
    fig.update_layout(
        title='Rich/Cheap Distribution by Tenor Bucket',
        yaxis_title='Rich/Cheap (bps)',
        height=500
    )
    fig.show()

## 4. Risk Analytics - PV01 Analysis

In [ ]:
if len(priced_df) > 0:
    # PV01 distribution by tenor
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=priced_df['Tenor_Years'],
        y=priced_df['pv01'].abs(),
        mode='markers',
        marker=dict(
            size=np.log10(priced_df['Notional amount-Leg 1'].clip(lower=1)) * 3,
            color='steelblue',
            opacity=0.6
        ),
        text=[f"Notional: ${n:,.0f}<br>PV01: ${p:,.0f}" 
              for n, p in zip(priced_df['Notional amount-Leg 1'], priced_df['pv01'])],
        hoverinfo='text+x+y'
    ))
    
    fig.update_layout(
        title='PV01 by Tenor (marker size = log notional)',
        xaxis_title='Tenor (Years)',
        yaxis_title='Absolute PV01 ($)',
        height=500
    )
    fig.show()

In [ ]:
if len(priced_df) > 0:
    # Aggregate risk by tenor bucket
    risk_by_tenor = priced_df.groupby('Tenor_Bucket').agg({
        'pv01': ['sum', 'mean', 'count'],
        'Notional amount-Leg 1': 'sum',
        'npv': 'sum'
    }).round(2)
    risk_by_tenor.columns = ['Total_PV01', 'Avg_PV01', 'Trade_Count', 'Total_Notional', 'Total_NPV']
    risk_by_tenor = risk_by_tenor.reindex([t for t in tenor_order if t in risk_by_tenor.index])
    
    print("Risk Summary by Tenor Bucket:")
    display(risk_by_tenor)

## 5. Compare Traded Rates to Curve

In [ ]:
if len(priced_df) > 0:
    # Traded rates vs fair curve
    fig = go.Figure()
    
    # Add traded rates as scatter
    fig.add_trace(go.Scatter(
        x=priced_df['Tenor_Years'],
        y=priced_df['Fixed rate-Leg 1'] * 100,
        mode='markers',
        name='Traded Rate',
        marker=dict(size=8, color='blue', opacity=0.5)
    ))
    
    # Add fair rates as a curve (sorted by tenor)
    sorted_df = priced_df.sort_values('Tenor_Years')
    fig.add_trace(go.Scatter(
        x=sorted_df['Tenor_Years'],
        y=sorted_df['fair_rate'] * 100,
        mode='lines',
        name='Fair Rate (Curve)',
        line=dict(color='red', width=2)
    ))
    
    fig.update_layout(
        title='Traded Rates vs Fair Value Curve',
        xaxis_title='Tenor (Years)',
        yaxis_title='Rate (%)',
        height=500,
        legend=dict(x=0.02, y=0.98)
    )
    fig.show()

## 6. Intraday Trading Pattern vs Fair Value

In [ ]:
if len(priced_df) > 0:
    # Rich/cheap evolution over time
    priced_df['Event timestamp'] = pd.to_datetime(priced_df['Event timestamp'])
    priced_df = priced_df.sort_values('Event timestamp')
    
    # Rolling average of rich/cheap
    priced_df['rolling_rich_cheap'] = priced_df['rich_cheap_bps'].rolling(window=10, min_periods=1).mean()
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Individual Trade Rich/Cheap', 'Rolling Average (10 trades)'])
    
    fig.add_trace(
        go.Scatter(x=priced_df['Event timestamp'], y=priced_df['rich_cheap_bps'],
                   mode='markers', name='Rich/Cheap',
                   marker=dict(size=6, color=priced_df['rich_cheap_bps'], 
                              colorscale='RdYlGn_r', showscale=False)),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=priced_df['Event timestamp'], y=priced_df['rolling_rich_cheap'],
                   mode='lines', name='Rolling Avg',
                   line=dict(color='blue', width=2)),
        row=2, col=1
    )
    
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)
    
    fig.update_layout(height=600, title_text='Rich/Cheap Evolution Throughout the Day')
    fig.show()

## 7. Summary Statistics

In [ ]:
if len(priced_df) > 0:
    print("="*60)
    print("PRICING SUMMARY")
    print("="*60)
    print(f"Trades Priced: {len(priced_df):,}")
    print(f"Total Notional: ${priced_df['Notional amount-Leg 1'].sum():,.0f}")
    print(f"\nRich/Cheap Analysis:")
    print(f"  Mean: {priced_df['rich_cheap_bps'].mean():.2f} bps")
    print(f"  Median: {priced_df['rich_cheap_bps'].median():.2f} bps")
    print(f"  Std Dev: {priced_df['rich_cheap_bps'].std():.2f} bps")
    print(f"  Min: {priced_df['rich_cheap_bps'].min():.2f} bps")
    print(f"  Max: {priced_df['rich_cheap_bps'].max():.2f} bps")
    print(f"\nRisk Metrics:")
    print(f"  Total Absolute PV01: ${priced_df['pv01'].abs().sum():,.0f}")
    print(f"  Total NPV: ${priced_df['npv'].sum():,.0f}")